In [2]:
import pandas as pd

df = pd.read_csv('data/old_sequences.csv', sep=";")
df

,id,sequence,gen,top10,pozycja_top100,points
0,1,TGCCTGGTTGACACGCTTGACGTTACCAAAACATTTCATATGGTGC...,0,0,0,0.0
1,2,AATGAAAGATGAGCTAAGAGATTATGGATATCAGACATGCCCTTTC...,0,0,0,0.0
2,3,AGGAACTGTCATCAAGATAATCGTGAAGAAAATCATACTTGACGGC...,0,0,0,0.0
3,4,TGTTGTATTATTTTGTTCTATGGAGCTTATAGATACGTTAATGCAA...,0,0,0,0.0
4,5,CTTTGTTCTCCATATTGCGTGGAGCTAGAGTTGCGGTATGTCTCAC...,0,0,0,0.0
...,...,...,...,...,...,...
2195,elite_06_003,TGTTGTTTTTTTTTCTTCTATGGAAATTAAAGATACGATAATTCAT...,7,4,1,17.0
2196,elite_06_004,TGTTGTATTTTTTTCTTCTATGGAAATTAAAGATACGATAATTCAT...,7,4,1,17.0
2197,elite_06_006,TGTTGTATTTTTTTCTTCTATGGAAATTAAAGATACGATAATTCAT...,7,4,1,17.0
2198,elite_06_007,TGTTGTATTTTTTTCTTCTATGGAAATTAAAGATACGATAATTCAT...,7,4,1,17.0


In [4]:
good = df[df['gen'] == 2]
good

,id,sequence,gen,top10,pozycja_top100,points
300,elite_01,TGTTGTATTTTTTTGTTCTATGGAGATTAACGATCCGATAATGCAA...,2,3,1,18.0
301,elite_04,TGTTGTATTTTTTTGTTCTATGGAGCTTAAAGATACGATAATGCAT...,2,3,1,18.0
302,elite_05,TGTTGTATTTTTTTGTTCTATGGAGCTTAAAGATACGATAATGCAT...,2,3,1,18.0
303,elite_07,TGTTGTATTTTTTTGTTCTATGGAGCTTATACATACGATAAAGCAT...,2,3,1,18.0
304,elite_09,TGTTGTATTTTTTTGTTCTATGGAGCTTAAAGATACGCTGATGCAT...,2,3,1,18.0
...,...,...,...,...,...,...
1695,elite_06_012,ATGTTTACTTTGTGGGCGCAGTATTTGTTATCACGAAAGATGTGCC...,2,3,1,18.0
1696,elite_06_014,ATGTTTACTTTGTGGCCTCGGTATTTGTTATAACGAAAGATGTGCC...,2,3,1,18.0
1697,elite_06_017,ATGTTTACTATGTGGGCGCAGTATTTGTTATAACGAAAGATGTACA...,2,3,1,18.0
1698,elite_06_019,ATGTTTACTATGTGGGCGCAGTATTTGTTATAACGAAAGATGTGCC...,2,3,1,18.0


In [5]:

import time
import traceback

import pandas as pd

from hack_the_bromoter.api import (
    MAX_SCORED_SEQUENCES,
    ApiError,
    build_fasta,
    check_sequence,
    get_client,
    me,
    me_all,
    nawigator_edycje,  # noqa: F401
    nawigator_mapa,  # noqa: F401
    ranking,  # noqa: F401
    wgraj,  # noqa: F401
    wild_sequence,  # noqa: F401
)
from hack_the_bromoter.judge import (
    Judge,
    bucket_sort_sequences,
    copeland_scores,  # noqa: F401
    sort_sequences,  # noqa: F401
)
from hack_the_bromoter.utils import (
    ID_COL,
    ROOT,
    SEQ_COL,
    convert_promoters,
    read_dataframe,
    save_dataframe,
    sequence_map,
    init_history
)

POPULATION = 100
GENERATIONS = 10000

# /wgraj is capped at one upload per 5 minutes *per key*; a generation can
# finish faster than that, so a submission that finds every key still cooling
# down is skipped rather than waited out (pass wait=True to sit it out).
UPLOAD_COOLDOWN_SLACK = 2.0

# The running table of everything we have tried, `;`-separated like the rest
# of the hackathon data -- read_dataframe/save_dataframe default to that, so
# never touch it with a bare pd.read_csv (which would default to `,`).
SEQUENCES_BACKLOG = ROOT / "sequences.csv"
PROMOTERS_SOURCE = ROOT / "HackThePromotor" / "Promotory.csv"

_START = time.monotonic()

from hack_the_bromoter.navigator import evolve

def log(message: str, indent: int = 0) -> None:
    """One timestamped progress line, flushed immediately."""
    print(f"[{time.monotonic() - _START:7.1f}s] {'  ' * indent}{message}", flush=True)

def submit(population, wait: bool = False) -> dict | None:
    """Upload the current population to /wgraj and print what it scored.

    Only the best submission of the day counts for the ranking, so sending
    every generation is free -- the one cost is the 5 minute upload cooldown
    of the key that gets spent. The cooldown is per key, so the upload goes
    out on whichever key is free; when none is, the submission is skipped
    (`wait=True` sleeps until the earliest one comes back instead) and the
    function returns None.
    """
    log(f"preparing a submission from {len(population)} rows", 1)

    mapping = sequence_map(population)
    if len(mapping) < len(population):
        log(f"{len(population) - len(mapping)} rows share an id and were "
            f"collapsed -- ids must be unique", 2)

    invalid = sum(1 for sequence in mapping.values() if check_sequence(sequence.upper()))
    duplicates = len(mapping) - len({s.upper() for s in mapping.values()})

    fasta = build_fasta(mapping)
    records = fasta.count(">")
    log(f"FASTA: {records} records kept of {len(mapping)}"
        f" ({invalid} rejected by check_sequence, {duplicates} duplicate sequences,"
        f" cap {MAX_SCORED_SEQUENCES})", 2)
    if not records:
        log("submission skipped: nothing survived the FASTA filters", 2)
        return None

    log("checking the upload cooldown on every key ...", 2)
    cooldowns = [account["zgloszenie_mozliwe_za_s"] for account in me_all()]
    log("cooldowns (s): " + ", ".join(f"key {n}={c:.0f}"
                                      for n, c in enumerate(cooldowns, 1)), 3)
    index = min(range(len(cooldowns)), key=cooldowns.__getitem__)
    if cooldowns[index] > 0:
        if not wait:
            log(f"submission skipped: every key is on the upload cooldown "
                f"({cooldowns[index]:.0f} s left on the earliest)", 2)
            return None
        log(f"waiting {cooldowns[index]:.0f} s for key {index + 1} "
            f"to come off the upload cooldown", 2)
        time.sleep(cooldowns[index] + UPLOAD_COOLDOWN_SLACK)

    log(f"POST /wgraj with {records} sequences on key {index + 1} ...", 2)
    started = time.monotonic()
    try:
        answer = get_client().wgraj(fasta, key_index=index)
    except ApiError as error:
        log(f"submission failed after {time.monotonic() - started:.1f} s: {error}", 2)
        return None

    log(f"submitted in {time.monotonic() - started:.1f} s:"
        f" scored {answer['ocenionych']}"
        f" | TOP10 {answer['pozycja_top10']}"
        f" | TOP100 {answer['pozycja_top100']}"
        f" | points {answer['punkty_razem']}", 2)
    if answer.get("filtrowanie"):
        log(f"server-side filtering: {answer['filtrowanie']}", 2)
    return answer


In [6]:
submit(good)

[    4.7s]   preparing a submission from 200 rows
[    4.7s]     38 rows share an id and were collapsed -- ids must be unique
[    4.7s]     FASTA: 100 records kept of 162 (0 rejected by check_sequence, 0 duplicate sequences, cap 100)
[    4.7s]     checking the upload cooldown on every key ...
[    5.3s]       cooldowns (s): key 1=0, key 2=87, key 3=0, key 4=0
[    5.3s]     POST /wgraj with 100 sequences on key 1 ...
[    5.7s]     submitted in 0.4 s: scored 100 | TOP10 4 | TOP100 2 | points 16.0
[    5.7s]     server-side filtering: {'n_w_pliku': 100, 'n_dlugosc_800': 100, 'n_unikalnych': 100, 'n_alfabet_ok': 100, 'n_po_filtrze_N': 100, 'n_ocenianych': 100, 'odrzucone_dlugosc': 0, 'odrzucone_duplikaty': 0, 'odrzucone_alfabet': 0, 'odrzucone_N': 0, 'prog_N': 'powyzej 80 z 800 zasad', 'pominiete_powyzej_limitu': 0, 'limit_ocenianych': 100, 'identyfikatory_odrzuconych': {}, 'identyfikatory_pominietych': []}


{'filtrowanie': {'n_w_pliku': 100,
  'n_dlugosc_800': 100,
  'n_unikalnych': 100,
  'n_alfabet_ok': 100,
  'n_po_filtrze_N': 100,
  'n_ocenianych': 100,
  'odrzucone_dlugosc': 0,
  'odrzucone_duplikaty': 0,
  'odrzucone_alfabet': 0,
  'odrzucone_N': 0,
  'prog_N': 'powyzej 80 z 800 zasad',
  'pominiete_powyzej_limitu': 0,
  'limit_ocenianych': 100,
  'identyfikatory_odrzuconych': {},
  'identyfikatory_pominietych': []},
 'ocenionych': 100,
 'dzielnik_top10': 10,
 'dzielnik_top100': 100,
 'pozycja_top10': 4,
 'pozycja_top100': 2,
 'punkty_razem': 16.0,
 'uwaga': 'Metryki i predykcje UKRYTE -- widzisz punkty i pozycje. TOP10 i TOP100 dziela ZAWSZE przez 10 i 100, wiec brakujace sekwencje wchodza jako zera. Przy wielu wgraniach liczy sie BIEZACE (po TOP10).'}